In [ ]:
# prep.py

from pathlib import Path

import cv2
import numpy as np
from PIL import Image


INPUT_FOLDER = Path("/data1/jaejung/others/hyeokjin/sam2/notebooks/Revision/1103_SD_and_SNR/Calibration/calib_Raw")
OUTPUT_FOLDER = Path("/data1/jaejung/others/hyeokjin/sam2/notebooks/Revision/1103_SD_and_SNR/Calibration/calib_gamma")
GAMMA_VALUE = 1.0


def adjust_gamma(image: np.ndarray, gamma: float = 1.0) -> np.ndarray:
    """Apply gamma correction to an 8-bit grayscale image."""
    if gamma <= 0:
        raise ValueError("gamma must be > 0")

    inv_gamma = 1.0 / gamma
    table = np.array(
        [((i / 255.0) ** inv_gamma) * 255 for i in range(256)],
        dtype=np.uint8,
    )
    return cv2.LUT(image, table)


def load_tif_as_uint16(image_path: Path) -> np.ndarray:
    """Load a tif image and keep its intensity depth."""
    image = Image.open(image_path).convert("I")
    return np.array(image, dtype=np.uint16)


def normalize_to_uint8(image: np.ndarray) -> np.ndarray:
    """Normalize image to 8-bit using the image max value."""
    max_val = int(image.max())
    if max_val == 0:
        return np.zeros_like(image, dtype=np.uint8)
    return ((image / max_val) * 255).astype(np.uint8)


def process_one_image(image_path: Path, output_dir: Path, gamma: float) -> None:
    print(f"Processing: {image_path.name}")

    image_u16 = load_tif_as_uint16(image_path)
    print(
        f"Before normalization: min={image_u16.min()}, "
        f"max={image_u16.max()}, dtype={image_u16.dtype}"
    )

    image_u8 = normalize_to_uint8(image_u16)
    print(
        f"After normalization: min={image_u8.min()}, "
        f"max={image_u8.max()}, dtype={image_u8.dtype}"
    )

    corrected = adjust_gamma(image_u8, gamma=gamma)
    print(
        f"After gamma correction: min={corrected.min()}, "
        f"max={corrected.max()}, dtype={corrected.dtype}"
    )

    output_path = output_dir / f"{image_path.stem}.png"
    cv2.imwrite(str(output_path), corrected)
    print(f"Saved: {output_path}")


def main() -> None:
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

    tif_files = sorted(INPUT_FOLDER.glob("*.tif"))
    if not tif_files:
        print(f"No .tif files found in: {INPUT_FOLDER}")
        return

    for image_path in tif_files:
        process_one_image(image_path, OUTPUT_FOLDER, GAMMA_VALUE)

    print("Gamma correction applied and images saved successfully.")


if __name__ == "__main__":
    main()